# 04 Alpha Walk-Forward Validation Engine

Validate constructed alpha candidates from Notebook 04A, including dynamic v3 alphas, using the existing walk-forward window and cross-sectional IC framework. This notebook writes dedicated constructed-alpha WFV tables and does not overwrite raw signal WFV outputs.


## 1. Purpose and scope

Run alpha-level WFV validation on constructed alpha candidates approved by `alpha_construction_quality_current`. This notebook is no longer a raw signal WFV workflow. It performs WFV validation only: no signal formula changes, alpha construction changes, stress testing, survivor freeze, portfolio construction, or ML.


## 2. Imports/config


In [1]:
from __future__ import annotations

import gc
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = next((path for path in [cwd, *cwd.parents] if (path / 'src').exists()), None)
if project_root is None:
    raise RuntimeError('Could not locate project root from current working directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.constructed_alpha_wfv import (
    apply_constructed_alpha_wfv_gate,
    build_constructed_alpha_wfv_failure_breakdown,
    build_constructed_alpha_wfv_winner_summary,
    load_constructed_alpha_candidates,
    run_constructed_alpha_wfv,
    summarize_constructed_alpha_wfv,
)
from src.constructed_alpha_wfv_storage import (
    CONSTRUCTED_ALPHA_WFV_TABLES,
    save_constructed_alpha_wfv_outputs,
)
from src.db import get_db_path, load_price_table, load_table
from src.run_config import make_run_id, make_run_timestamp
from src.walkforward import generate_walkforward_windows

CONSTRUCTED_ALPHA_WFV_VERSION = 'phase4_constructed_alpha_wfv_v1'
HORIZONS = [1, 5, 10, 20]
TRAIN_SIZE = 378
TEST_SIZE = 63
PURGE_SIZE = 20
EMBARGO_SIZE = 5
IC_METHOD = 'spearman'

sqlite_db_path = get_db_path()
print(f'SQLite database: {sqlite_db_path}')
print(f'Constructed alpha WFV version: {CONSTRUCTED_ALPHA_WFV_VERSION}')


SQLite database: /Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db
Constructed alpha WFV version: phase4_constructed_alpha_wfv_v1


## 3. Create run_id/timestamp


In [2]:
run_id = make_run_id('constructed_alpha_wfv')
run_timestamp = make_run_timestamp()

print(f'run_id: {run_id}')
print(f'run_timestamp: {run_timestamp}')


run_id: constructed_alpha_wfv_20260511_081122
run_timestamp: 2026-05-11 08:11:22


## 4. Load approved constructed alphas


In [3]:
approved_constructed_alphas = load_constructed_alpha_candidates(db_path=sqlite_db_path)
if approved_constructed_alphas.empty:
    print('No constructed alpha candidates approved for alpha validation; writing empty 04B WFV outputs for lineage consistency.')
else:
    approved_alpha_names = approved_constructed_alphas['alpha_name'].dropna().astype(str).tolist()
    print(f'Approved constructed alpha candidates from alpha_construction_quality_current: {approved_alpha_names}')

approved_alpha_names = approved_constructed_alphas['alpha_name'].dropna().astype(str).tolist() if not approved_constructed_alphas.empty else []
display(approved_constructed_alphas)


Approved constructed alpha candidates from alpha_construction_quality_current: ['alpha_decay_aware_dynamic_v4_smooth', 'alpha_hybrid_adaptive_v4_smooth', 'alpha_orthogonal_diversifier_v1_smooth', 'alpha_orthogonal_diversifier_v2_score_weighted_smooth', 'alpha_regime_blend_dynamic_v4_smooth', 'alpha_rolling_ic_dynamic_v4_smooth']


,alpha_name,finite_pct,missing_pct,max_abs_alpha,avg_turnover_proxy,n_dates,n_tickers,first_valid_date,last_valid_date,status,quality_notes,run_id,alpha_construction_version
0,alpha_decay_aware_dynamic_v4_smooth,0.995625,0.004375,3.000000,1.737177,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
1,alpha_hybrid_adaptive_v4_smooth,0.995486,0.004514,3.000000,1.738334,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
2,alpha_orthogonal_diversifier_v1_smooth,0.991994,0.008006,3.000000,1.683714,2098,478,2018-04-06,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
3,alpha_orthogonal_diversifier_v2_score_weighted...,0.995255,0.004745,2.811819,1.946872,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
4,alpha_regime_blend_dynamic_v4_smooth,0.995625,0.004375,3.000000,1.746788,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
5,alpha_rolling_ic_dynamic_v4_smooth,0.995486,0.004514,3.000000,1.743410,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4


## 5. Load alpha_constructed_candidates_current


In [4]:
alpha_long_all = load_table('alpha_constructed_candidates_current', db_path=sqlite_db_path)
alpha_names_found_in_long_table = sorted(alpha_long_all['alpha_name'].dropna().astype(str).unique().tolist()) if 'alpha_name' in alpha_long_all.columns else []
approved_alpha_names = approved_constructed_alphas['alpha_name'].dropna().astype(str).tolist() if not approved_constructed_alphas.empty else []
approved_alpha_names_found = [name for name in approved_alpha_names if name in alpha_names_found_in_long_table]
approved_alpha_names_missing = sorted(set(approved_alpha_names).difference(alpha_names_found_in_long_table))

alpha_long = alpha_long_all.loc[alpha_long_all['alpha_name'].isin(approved_alpha_names_found)].copy() if approved_alpha_names_found else pd.DataFrame(columns=alpha_long_all.columns)
if not alpha_long.empty:
    alpha_long['Date'] = pd.to_datetime(alpha_long['Date'], errors='coerce')
    alpha_long['alpha_value'] = pd.to_numeric(alpha_long['alpha_value'], errors='coerce')
alpha_names_sent_to_wfv = sorted(alpha_long['alpha_name'].dropna().astype(str).unique().tolist()) if 'alpha_name' in alpha_long.columns else []

alpha_candidate_validation = pd.DataFrame(
    [
        {'check': 'approved_from_quality', 'alpha_names': approved_alpha_names, 'n_alpha_names': len(approved_alpha_names)},
        {'check': 'found_in_alpha_long_table', 'alpha_names': alpha_names_found_in_long_table, 'n_alpha_names': len(alpha_names_found_in_long_table)},
        {'check': 'sent_to_wfv', 'alpha_names': alpha_names_sent_to_wfv, 'n_alpha_names': len(alpha_names_sent_to_wfv)},
        {'check': 'approved_missing_from_long_table', 'alpha_names': approved_alpha_names_missing, 'n_alpha_names': len(approved_alpha_names_missing)},
    ]
)
alpha_input_shape = pd.DataFrame([
    {'input_name': 'alpha_constructed_candidates_current_filtered', 'rows': len(alpha_long), 'columns': len(alpha_long.columns)}
])

if approved_alpha_names_missing:
    raise ValueError(f'Approved alpha candidates missing from alpha_constructed_candidates_current: {approved_alpha_names_missing}')
if approved_alpha_names and sorted(approved_alpha_names) != alpha_names_sent_to_wfv:
    raise ValueError('Mismatch between approved alpha candidates and alpha names sent to WFV.')

display(alpha_candidate_validation)
display(alpha_input_shape)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after loading filtered alpha candidates for WFV')


,check,alpha_names,n_alpha_names
0,approved_from_quality,"[alpha_decay_aware_dynamic_v4_smooth, alpha_hy...",6
1,found_in_alpha_long_table,"[alpha_decay_aware_dynamic_v3, alpha_decay_awa...",10
2,sent_to_wfv,"[alpha_decay_aware_dynamic_v4_smooth, alpha_hy...",6
3,approved_missing_from_long_table,[],0


,input_name,rows,columns
0,alpha_constructed_candidates_current_filtered,6017064,6


## 6. Load clean close prices / forward returns


In [5]:
close_prices = load_price_table('clean_close_prices_current', db_path=sqlite_db_path)
print(f'Close price panel shape: {close_prices.shape}')
display(close_prices.tail())


Close price panel shape: (2098, 478)


,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WTW,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2026-05-01,NaN,280.140015,206.600006,141.660004,89.459999,NaN,179.830002,250.710007,397.690002,NaN,...,NaN,NaN,NaN,82.580002,152.750000,115.370003,NaN,NaN,NaN,114.160004
2026-05-04,NaN,276.829987,208.160004,138.860001,87.540001,NaN,180.119995,253.960007,397.019989,NaN,...,NaN,NaN,NaN,81.169998,153.690002,114.839996,NaN,NaN,NaN,112.680000
2026-05-05,NaN,284.179993,206.110001,139.729996,87.169998,NaN,179.009995,255.619995,404.769989,NaN,...,NaN,NaN,NaN,81.449997,154.880005,116.389999,NaN,NaN,NaN,112.540001
2026-05-06,NaN,287.510010,205.029999,139.880005,86.300003,NaN,174.570007,250.169998,415.630005,NaN,...,NaN,NaN,NaN,80.550003,148.690002,118.589996,NaN,NaN,NaN,111.220001
2026-05-07,NaN,287.440002,202.710007,140.460007,87.010002,NaN,180.190002,256.510010,408.519989,NaN,...,NaN,NaN,NaN,80.430000,146.580002,115.639999,NaN,NaN,NaN,87.309998


## 7. Generate WFV windows


In [6]:
wfv_config = pd.DataFrame([
    {'parameter': 'horizons', 'value': str(HORIZONS)},
    {'parameter': 'train_size', 'value': TRAIN_SIZE},
    {'parameter': 'test_size', 'value': TEST_SIZE},
    {'parameter': 'purge_size', 'value': PURGE_SIZE},
    {'parameter': 'embargo_size', 'value': EMBARGO_SIZE},
    {'parameter': 'ic_method', 'value': IC_METHOD},
    {'parameter': 'constructed_alpha_wfv_version', 'value': CONSTRUCTED_ALPHA_WFV_VERSION},
])
display(wfv_config)

windows = generate_walkforward_windows(
    close_prices.index,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    purge_size=PURGE_SIZE,
    embargo_size=EMBARGO_SIZE,
)
if windows.empty:
    raise ValueError('WFV configuration produced no windows.')
window_count = len(windows)
print(f'Generated WFV windows: {window_count}')
display(windows)


,parameter,value
0,horizons,"[1, 5, 10, 20]"
1,train_size,378
2,test_size,63
3,purge_size,20
4,embargo_size,5
5,ic_method,spearman
6,constructed_alpha_wfv_version,phase4_constructed_alpha_wfv_v1


Generated WFV windows: 4


,window_id,train_start,train_end,test_start,test_end,purge_size,embargo_size,embargo_start,embargo_end,n_train_dates,n_test_dates
0,1,2018-01-02,2019-07-03,2019-08-02,2019-10-30,20,5,2019-10-31,2019-11-06,378,63
1,2,2019-11-07,2021-05-10,2021-06-09,2021-09-07,20,5,2021-09-08,2021-09-14,378,63
2,3,2021-09-15,2023-03-16,2023-04-17,2023-07-17,20,5,2023-07-18,2023-07-24,378,63
3,4,2023-07-25,2025-01-24,2025-02-25,2025-05-23,20,5,2025-05-27,2025-06-02,378,63


## 8. Run constructed alpha WFV


In [7]:
constructed_alpha_wfv_window_results = run_constructed_alpha_wfv(
    approved_alphas=approved_constructed_alphas,
    alpha_long_df=alpha_long,
    close_prices=close_prices,
    windows=windows,
    horizons=HORIZONS,
    method=IC_METHOD,
)

print(f'WFV window result shape: {constructed_alpha_wfv_window_results.shape}')
display(constructed_alpha_wfv_window_results.head())


WFV window result shape: (96, 15)


,window_id,alpha_name,horizon,method,train_start,train_end,test_start,test_end,train_mean_ic,test_mean_ic,train_positive_ic_rate,test_positive_ic_rate,train_n_obs,test_n_obs,expected_direction
0,1,alpha_decay_aware_dynamic_v4_smooth,1,spearman,2018-01-02,2019-07-03,2019-08-02,2019-10-30,0.011537,-0.005255,0.528529,0.460317,97613,18634,POSITIVE
1,2,alpha_decay_aware_dynamic_v4_smooth,1,spearman,2019-11-07,2021-05-10,2021-06-09,2021-09-07,-0.003944,0.011476,0.473545,0.523810,112421,18765,POSITIVE
2,3,alpha_decay_aware_dynamic_v4_smooth,1,spearman,2021-09-15,2023-03-16,2023-04-17,2023-07-17,-0.004224,-0.010991,0.500000,0.428571,112804,18782,POSITIVE
3,4,alpha_decay_aware_dynamic_v4_smooth,1,spearman,2023-07-25,2025-01-24,2025-02-25,2025-05-23,0.004140,0.025124,0.542328,0.539683,112711,18778,POSITIVE
4,1,alpha_decay_aware_dynamic_v4_smooth,5,spearman,2018-01-02,2019-07-03,2019-08-02,2019-10-30,0.019345,-0.016667,0.582583,0.333333,95784,18229,POSITIVE


## 9. Build summary/gate/failure breakdown/winner summary


In [8]:
constructed_alpha_wfv_summary = summarize_constructed_alpha_wfv(constructed_alpha_wfv_window_results)
constructed_alpha_wfv_gate = apply_constructed_alpha_wfv_gate(constructed_alpha_wfv_summary)
constructed_alpha_wfv_failure_breakdown = build_constructed_alpha_wfv_failure_breakdown(constructed_alpha_wfv_gate)
constructed_alpha_wfv_winner_summary = build_constructed_alpha_wfv_winner_summary(constructed_alpha_wfv_gate)

wfv_gate_counts = (
    constructed_alpha_wfv_gate['status'].value_counts()
    if 'status' in constructed_alpha_wfv_gate.columns
    else pd.Series(dtype='int64')
)

display(constructed_alpha_wfv_summary)
display(constructed_alpha_wfv_gate)
display(constructed_alpha_wfv_failure_breakdown)
display(constructed_alpha_wfv_winner_summary)


,alpha_name,horizon,n_windows,mean_train_ic,mean_test_ic,median_test_ic,test_ic_std,test_ic_ir,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,n_positive_test_windows,n_negative_test_windows
0,alpha_decay_aware_dynamic_v4_smooth,1,4,0.001877,0.005088,0.003110,0.016409,0.310096,0.005088,0.310096,0.50,0.50,2,2
1,alpha_decay_aware_dynamic_v4_smooth,5,4,0.004471,0.013027,0.000707,0.039157,0.332675,0.013027,0.332675,0.25,0.50,2,2
2,alpha_decay_aware_dynamic_v4_smooth,10,4,0.006950,0.022402,0.003008,0.057291,0.391022,0.022402,0.391022,0.25,0.50,2,2
3,alpha_decay_aware_dynamic_v4_smooth,20,4,0.004645,0.044253,0.019681,0.069003,0.641317,0.044253,0.641317,0.25,0.75,3,1
4,alpha_hybrid_adaptive_v4_smooth,1,4,0.001833,0.000979,-0.001766,0.015887,0.061645,0.000979,0.061645,0.50,0.50,2,2
5,alpha_hybrid_adaptive_v4_smooth,5,4,0.004562,0.005697,-0.008045,0.037866,0.150449,0.005697,0.150449,0.25,0.50,2,2
6,alpha_hybrid_adaptive_v4_smooth,10,4,0.007202,0.013060,-0.008902,0.057013,0.229074,0.013060,0.229074,0.25,0.50,2,2
7,alpha_hybrid_adaptive_v4_smooth,20,4,0.005365,0.034611,0.010339,0.068111,0.508163,0.034611,0.508163,0.50,0.75,3,1
8,alpha_orthogonal_diversifier_v1_smooth,1,4,-0.000535,0.006003,0.006635,0.026702,0.224823,0.006003,0.224823,0.25,0.50,2,2
9,alpha_orthogonal_diversifier_v1_smooth,5,4,-0.001958,0.010004,0.014272,0.053887,0.185638,0.010004,0.185638,0.25,0.50,2,2


,alpha_name,horizon,n_windows,mean_train_ic,mean_test_ic,median_test_ic,test_ic_std,test_ic_ir,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,n_positive_test_windows,n_negative_test_windows,status,constructed_alpha_wfv_notes
0,alpha_decay_aware_dynamic_v4_smooth,1,4,0.001877,0.005088,0.003110,0.016409,0.310096,0.005088,0.310096,0.50,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low sign consistency
1,alpha_decay_aware_dynamic_v4_smooth,5,4,0.004471,0.013027,0.000707,0.039157,0.332675,0.013027,0.332675,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency
2,alpha_decay_aware_dynamic_v4_smooth,10,4,0.006950,0.022402,0.003008,0.057291,0.391022,0.022402,0.391022,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency
3,alpha_decay_aware_dynamic_v4_smooth,20,4,0.004645,0.044253,0.019681,0.069003,0.641317,0.044253,0.641317,0.25,0.75,3,1,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence
4,alpha_hybrid_adaptive_v4_smooth,1,4,0.001833,0.000979,-0.001766,0.015887,0.061645,0.000979,0.061645,0.50,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low sign consistency
5,alpha_hybrid_adaptive_v4_smooth,5,4,0.004562,0.005697,-0.008045,0.037866,0.150449,0.005697,0.150449,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low persistence; low sign c...
6,alpha_hybrid_adaptive_v4_smooth,10,4,0.007202,0.013060,-0.008902,0.057013,0.229074,0.013060,0.229074,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency
7,alpha_hybrid_adaptive_v4_smooth,20,4,0.005365,0.034611,0.010339,0.068111,0.508163,0.034611,0.508163,0.50,0.75,3,1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,Meets secondary constructed alpha WFV thresholds.
8,alpha_orthogonal_diversifier_v1_smooth,1,4,-0.000535,0.006003,0.006635,0.026702,0.224823,0.006003,0.224823,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low persistence; low sign c...
9,alpha_orthogonal_diversifier_v1_smooth,5,4,-0.001958,0.010004,0.014272,0.053887,0.185638,0.010004,0.185638,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency


,failure_reason,count,pct_of_rejected
0,low sign consistency,18,0.947368
1,weak effective IC,11,0.578947
2,low persistence,10,0.526316
3,weak effective IC IR,6,0.315789


,alpha_name,horizon,status,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,constructed_alpha_wfv_notes
0,alpha_orthogonal_diversifier_v2_score_weighted...,20,APPROVED_CONSTRUCTED_ALPHA_WFV,0.037789,0.739775,1.00,0.75,Meets strict constructed alpha WFV thresholds.
1,alpha_decay_aware_dynamic_v4_smooth,20,REJECTED_CONSTRUCTED_ALPHA_WFV,0.044253,0.641317,0.25,0.75,low persistence
2,alpha_orthogonal_diversifier_v1_smooth,20,REJECTED_CONSTRUCTED_ALPHA_WFV,0.012772,0.129458,0.25,0.50,low persistence; low sign consistency
3,alpha_rolling_ic_dynamic_v4_smooth,20,REJECTED_CONSTRUCTED_ALPHA_WFV,0.002863,0.045922,0.50,0.25,weak effective IC; weak effective IC IR; low s...
4,alpha_regime_blend_dynamic_v4_smooth,20,WATCHLIST_CONSTRUCTED_ALPHA_WFV,0.046854,0.726320,0.50,0.75,Meets secondary constructed alpha WFV thresholds.
5,alpha_hybrid_adaptive_v4_smooth,20,WATCHLIST_CONSTRUCTED_ALPHA_WFV,0.034611,0.508163,0.50,0.75,Meets secondary constructed alpha WFV thresholds.


## 10. Save outputs to SQLite


In [9]:
saved_paths = save_constructed_alpha_wfv_outputs(
    windows=windows,
    window_results=constructed_alpha_wfv_window_results,
    summary=constructed_alpha_wfv_summary,
    gate=constructed_alpha_wfv_gate,
    failure_breakdown=constructed_alpha_wfv_failure_breakdown,
    winner_summary=constructed_alpha_wfv_winner_summary,
    db_path=sqlite_db_path,
    run_id=run_id,
    constructed_alpha_wfv_version=CONSTRUCTED_ALPHA_WFV_VERSION,
)

sqlite_tables_written = pd.DataFrame([
    {
        'artifact': artifact,
        'current_table': tables[0],
        'history_table': tables[1],
        'sqlite_path': str(saved_paths[artifact]),
    }
    for artifact, tables in CONSTRUCTED_ALPHA_WFV_TABLES.items()
])

display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving constructed alpha WFV outputs')


,artifact,current_table,history_table,sqlite_path
0,windows,constructed_alpha_wfv_windows_current,constructed_alpha_wfv_windows_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,window_results,constructed_alpha_wfv_window_results_current,constructed_alpha_wfv_window_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,summary,constructed_alpha_wfv_summary_current,constructed_alpha_wfv_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,gate,constructed_alpha_wfv_gate_current,constructed_alpha_wfv_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,failure_breakdown,constructed_alpha_wfv_failure_breakdown_current,constructed_alpha_wfv_failure_breakdown_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,winner_summary,constructed_alpha_wfv_winner_summary_current,constructed_alpha_wfv_winner_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 11. Final display


In [10]:
print('Approved constructed alpha candidates')
display(approved_constructed_alphas)

print('Alpha candidate input validation')
display(alpha_candidate_validation)

print('Window count')
display(pd.DataFrame([{'window_count': window_count}]))

print('WFV gate counts')
display(wfv_gate_counts.rename('alpha_horizon_count'))

print('Winner summary')
display(constructed_alpha_wfv_winner_summary)

print('Failure breakdown')
display(constructed_alpha_wfv_failure_breakdown)

print('Full gate table')
display(constructed_alpha_wfv_gate)

print('SQLite tables written')
display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving constructed alpha WFV outputs')


Approved constructed alpha candidates


,alpha_name,finite_pct,missing_pct,max_abs_alpha,avg_turnover_proxy,n_dates,n_tickers,first_valid_date,last_valid_date,status,quality_notes,run_id,alpha_construction_version
0,alpha_decay_aware_dynamic_v4_smooth,0.995625,0.004375,3.000000,1.737177,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
1,alpha_hybrid_adaptive_v4_smooth,0.995486,0.004514,3.000000,1.738334,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
2,alpha_orthogonal_diversifier_v1_smooth,0.991994,0.008006,3.000000,1.683714,2098,478,2018-04-06,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
3,alpha_orthogonal_diversifier_v2_score_weighted...,0.995255,0.004745,2.811819,1.946872,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
4,alpha_regime_blend_dynamic_v4_smooth,0.995625,0.004375,3.000000,1.746788,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4
5,alpha_rolling_ic_dynamic_v4_smooth,0.995486,0.004514,3.000000,1.743410,2098,478,2018-03-29,2026-05-07,APPROVED_FOR_ALPHA_VALIDATION,"Passes construction coverage, scale, and turno...",phase4a_alpha_construction_20260511_075950,phase4a_alpha_construction_v4


Alpha candidate input validation


,check,alpha_names,n_alpha_names
0,approved_from_quality,"[alpha_decay_aware_dynamic_v4_smooth, alpha_hy...",6
1,found_in_alpha_long_table,"[alpha_decay_aware_dynamic_v3, alpha_decay_awa...",10
2,sent_to_wfv,"[alpha_decay_aware_dynamic_v4_smooth, alpha_hy...",6
3,approved_missing_from_long_table,[],0


Window count


...

,failure_reason,count,pct_of_rejected
0,low sign consistency,18,0.947368
1,weak effective IC,11,0.578947
2,low persistence,10,0.526316
3,weak effective IC IR,6,0.315789


Full gate table


,alpha_name,horizon,n_windows,mean_train_ic,mean_test_ic,median_test_ic,test_ic_std,test_ic_ir,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,n_positive_test_windows,n_negative_test_windows,status,constructed_alpha_wfv_notes
0,alpha_decay_aware_dynamic_v4_smooth,1,4,0.001877,0.005088,0.003110,0.016409,0.310096,0.005088,0.310096,0.50,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low sign consistency
1,alpha_decay_aware_dynamic_v4_smooth,5,4,0.004471,0.013027,0.000707,0.039157,0.332675,0.013027,0.332675,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency
2,alpha_decay_aware_dynamic_v4_smooth,10,4,0.006950,0.022402,0.003008,0.057291,0.391022,0.022402,0.391022,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency
3,alpha_decay_aware_dynamic_v4_smooth,20,4,0.004645,0.044253,0.019681,0.069003,0.641317,0.044253,0.641317,0.25,0.75,3,1,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence
4,alpha_hybrid_adaptive_v4_smooth,1,4,0.001833,0.000979,-0.001766,0.015887,0.061645,0.000979,0.061645,0.50,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low sign consistency
5,alpha_hybrid_adaptive_v4_smooth,5,4,0.004562,0.005697,-0.008045,0.037866,0.150449,0.005697,0.150449,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low persistence; low sign c...
6,alpha_hybrid_adaptive_v4_smooth,10,4,0.007202,0.013060,-0.008902,0.057013,0.229074,0.013060,0.229074,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency
7,alpha_hybrid_adaptive_v4_smooth,20,4,0.005365,0.034611,0.010339,0.068111,0.508163,0.034611,0.508163,0.50,0.75,3,1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,Meets secondary constructed alpha WFV thresholds.
8,alpha_orthogonal_diversifier_v1_smooth,1,4,-0.000535,0.006003,0.006635,0.026702,0.224823,0.006003,0.224823,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low persistence; low sign c...
9,alpha_orthogonal_diversifier_v1_smooth,5,4,-0.001958,0.010004,0.014272,0.053887,0.185638,0.010004,0.185638,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,windows,constructed_alpha_wfv_windows_current,constructed_alpha_wfv_windows_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,window_results,constructed_alpha_wfv_window_results_current,constructed_alpha_wfv_window_results_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,summary,constructed_alpha_wfv_summary_current,constructed_alpha_wfv_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,gate,constructed_alpha_wfv_gate_current,constructed_alpha_wfv_gate_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,failure_breakdown,constructed_alpha_wfv_failure_breakdown_current,constructed_alpha_wfv_failure_breakdown_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,winner_summary,constructed_alpha_wfv_winner_summary_current,constructed_alpha_wfv_winner_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


In [11]:
from src.db import load_table

gate = load_table("constructed_alpha_wfv_gate_current")
winner = load_table("constructed_alpha_wfv_winner_summary_current")

if "status" in gate.columns:
    display(gate["status"].value_counts())
else:
    print("constructed_alpha_wfv_gate_current is empty for this run.")
if not winner.empty and {"status", "effective_mean_test_ic"}.issubset(winner.columns):
    display(winner.sort_values(["status", "effective_mean_test_ic"], ascending=[True, False]))
else:
    print("constructed_alpha_wfv_winner_summary_current is empty for this run.")
if not gate.empty and "alpha_name" in gate.columns:
    display(
        gate[gate["alpha_name"].str.contains("dynamic|rolling|hybrid", case=False, na=False)]
        .sort_values(["alpha_name", "horizon"])
    )


status
REJECTED_CONSTRUCTED_ALPHA_WFV     19
WATCHLIST_CONSTRUCTED_ALPHA_WFV     4
APPROVED_CONSTRUCTED_ALPHA_WFV      1
Name: count, dtype: int64

,alpha_name,horizon,status,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,constructed_alpha_wfv_notes,run_id,constructed_alpha_wfv_version
0,alpha_orthogonal_diversifier_v2_score_weighted...,20,APPROVED_CONSTRUCTED_ALPHA_WFV,0.037789,0.739775,1.00,0.75,Meets strict constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
1,alpha_decay_aware_dynamic_v4_smooth,20,REJECTED_CONSTRUCTED_ALPHA_WFV,0.044253,0.641317,0.25,0.75,low persistence,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
2,alpha_orthogonal_diversifier_v1_smooth,20,REJECTED_CONSTRUCTED_ALPHA_WFV,0.012772,0.129458,0.25,0.50,low persistence; low sign consistency,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
3,alpha_rolling_ic_dynamic_v4_smooth,20,REJECTED_CONSTRUCTED_ALPHA_WFV,0.002863,0.045922,0.50,0.25,weak effective IC; weak effective IC IR; low s...,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
4,alpha_regime_blend_dynamic_v4_smooth,20,WATCHLIST_CONSTRUCTED_ALPHA_WFV,0.046854,0.726320,0.50,0.75,Meets secondary constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
5,alpha_hybrid_adaptive_v4_smooth,20,WATCHLIST_CONSTRUCTED_ALPHA_WFV,0.034611,0.508163,0.50,0.75,Meets secondary constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1


,alpha_name,horizon,n_windows,mean_train_ic,mean_test_ic,median_test_ic,test_ic_std,test_ic_ir,effective_mean_test_ic,effective_test_ic_ir,persistence_ratio,sign_consistency,n_positive_test_windows,n_negative_test_windows,status,constructed_alpha_wfv_notes,run_id,constructed_alpha_wfv_version
0,alpha_decay_aware_dynamic_v4_smooth,1,4,0.001877,0.005088,0.003110,0.016409,0.310096,0.005088,0.310096,0.50,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low sign consistency,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
1,alpha_decay_aware_dynamic_v4_smooth,5,4,0.004471,0.013027,0.000707,0.039157,0.332675,0.013027,0.332675,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
2,alpha_decay_aware_dynamic_v4_smooth,10,4,0.006950,0.022402,0.003008,0.057291,0.391022,0.022402,0.391022,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
3,alpha_decay_aware_dynamic_v4_smooth,20,4,0.004645,0.044253,0.019681,0.069003,0.641317,0.044253,0.641317,0.25,0.75,3,1,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
4,alpha_hybrid_adaptive_v4_smooth,1,4,0.001833,0.000979,-0.001766,0.015887,0.061645,0.000979,0.061645,0.50,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low sign consistency,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
5,alpha_hybrid_adaptive_v4_smooth,5,4,0.004562,0.005697,-0.008045,0.037866,0.150449,0.005697,0.150449,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low persistence; low sign c...,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
6,alpha_hybrid_adaptive_v4_smooth,10,4,0.007202,0.013060,-0.008902,0.057013,0.229074,0.013060,0.229074,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
7,alpha_hybrid_adaptive_v4_smooth,20,4,0.005365,0.034611,0.010339,0.068111,0.508163,0.034611,0.508163,0.50,0.75,3,1,WATCHLIST_CONSTRUCTED_ALPHA_WFV,Meets secondary constructed alpha WFV thresholds.,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
16,alpha_regime_blend_dynamic_v4_smooth,1,4,0.002199,0.005305,0.002830,0.015853,0.334607,0.005305,0.334607,0.50,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,weak effective IC; low sign consistency,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
17,alpha_regime_blend_dynamic_v4_smooth,5,4,0.005231,0.013784,0.003505,0.036816,0.374404,0.013784,0.374404,0.25,0.50,2,2,REJECTED_CONSTRUCTED_ALPHA_WFV,low persistence; low sign consistency,constructed_alpha_wfv_20260511_081122,phase4_constructed_alpha_wfv_v1
